# Handwritten Digit and Symbol Recognition
Assignment 1 — self-collected images, logistic regression baseline, convolutional neural network.

This is a starter experiment, not a completed submission. Collect real images before running. Results are generated by execution; none are invented. Folder names define classes, so digits and math symbols are supported.

Use Python 3.11 or 3.12 in Jupyter, or upload the notebook and dataset to Google Colab. On Colab, place the dataset under `/content/data/raw` (or change ROOT).

In [ ]:
%pip install numpy pandas pillow matplotlib scikit-learn torch torchvision joblib

## 1. Configuration and reproducibility
Store images as `data/raw/0/w01_001.png`, `data/raw/plus/w01_001.png`, etc. The part before the first underscore is the writer ID. Use the same ID for that writer in EVERY class. Each image contains one tightly cropped, complete character on a light background. Reserve writers w15–w17 for validation and w18–w20 for testing in the suggested 20-writer collection. Change the sets below if needed, before training.

In [ ]:
from pathlib import Path
import copy, hashlib, json, random
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, ConfusionMatrixDisplay
import joblib

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
ROOT = Path('data/raw')
OUT = Path('results'); OUT.mkdir(exist_ok=True)
VAL_WRITERS = {'w15', 'w16', 'w17'}
TEST_WRITERS = {'w18', 'w19', 'w20'}
assert not VAL_WRITERS & TEST_WRITERS
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## 2. Audit and split the original images
Writer-disjoint splits measure recognition of unseen handwriting. Never split augmented copies independently. Exact duplicate pixels are rejected; manually inspect for near-duplicates, cropped-off strokes, and label errors too. The assignment says 100–200 images per class; aim for 200 original examples per class.

In [ ]:
assert ROOT.is_dir(), f'Collect your images and create {ROOT.resolve()} first.'
classes = sorted(p.name for p in ROOT.iterdir() if p.is_dir())
assert len(classes) >= 2, 'At least two class folders are required.'
records, seen = [], {}
for label, name in enumerate(classes):
    for path in sorted((ROOT/name).iterdir()):
        if path.suffix.lower() not in {'.png', '.jpg', '.jpeg', '.bmp'}: continue
        assert '_' in path.stem, f'Missing writer ID: {path}'
        writer = path.stem.split('_')[0]
        with Image.open(path) as im:
            im = ImageOps.exif_transpose(im).convert('L')
            digest = hashlib.sha256(str(im.size).encode()+im.tobytes()).hexdigest()
        assert digest not in seen, f'Duplicate image: {path} and {seen.get(digest)}'
        seen[digest] = str(path)
        split = 'test' if writer in TEST_WRITERS else 'val' if writer in VAL_WRITERS else 'train'
        records.append(dict(path=str(path), label=label, class_name=name, writer=writer, split=split))
df = pd.DataFrame(records)
assert len(df), 'No images found.'
counts = pd.crosstab(df.class_name, df.split).reindex(index=classes, columns=['train','val','test'], fill_value=0)
display(counts)
assert (counts > 0).all().all(), 'Every class must occur in each split. Check writer IDs.'
assert (counts.sum(axis=1) >= 100).all(), 'Collect at least 100 original images per class; target 200.'
df.to_csv(OUT/'dataset_manifest.csv', index=False)
counts.to_csv(OUT/'dataset_counts.csv')
(OUT/'classes.json').write_text(json.dumps(classes), encoding='utf-8')
parts = {s: df[df.split == s].reset_index(drop=True) for s in ['train','val','test']}
print('Writers by split:', {s: sorted(p.writer.unique()) for s,p in parts.items()})

## 3. Preprocessing and augmentation
Convert to grayscale, invert dark ink to a bright foreground, preserve aspect ratio while fitting into 24×24 pixels, and pad to 32×32. Scale pixels to [0,1]. Training-only augmentation uses small rotations and translations. Avoid horizontal/vertical flips, which can change the character's meaning. Both models receive originals plus one fixed augmented training copy so their comparison uses the same input data.

In [ ]:
def preprocess(path):
    with Image.open(path) as raw:
        im = ImageOps.invert(ImageOps.exif_transpose(raw).convert('L'))
        im = ImageOps.contain(im, (24,24), method=Image.Resampling.LANCZOS)
        canvas = Image.new('L', (32,32), 0)
        canvas.paste(im, ((32-im.width)//2, (32-im.height)//2))
        return canvas

augment = transforms.RandomAffine(degrees=10, translate=(0.08,0.08), fill=0)
to_tensor = transforms.ToTensor()
def tensors(frame, augmented=False):
    images = [preprocess(p) for p in frame.path]
    X = torch.stack([to_tensor(im) for im in images])
    y = torch.tensor(frame.label.to_numpy(), dtype=torch.long)
    if augmented:
        X = torch.cat([X, torch.stack([to_tensor(augment(im)) for im in images])])
        y = torch.cat([y,y])
    return X,y

Xtrain,ytrain = tensors(parts['train'], augmented=True)
Xval,yval = tensors(parts['val'])
# Test pixels are loaded only in the final evaluation section.
fig, axes = plt.subplots(2,8,figsize=(12,3))
for ax,i in zip(axes.flat, range(min(16,len(parts['train'])))):
    ax.imshow(Xtrain[i,0],cmap='gray'); ax.set_title(classes[ytrain[i].item()]); ax.axis('off')
plt.tight_layout(); plt.savefig(OUT/'training_examples.png', dpi=180); plt.show()

## 4. Baseline: logistic regression
Flatten images into 1,024 features. Select regularization using validation macro F1 only. Macro F1 weights each class equally.

In [ ]:
baseline, best_score, baseline_trials = None, -1, []
for C in [0.1, 1.0, 10.0]:
    model = LogisticRegression(C=C, max_iter=2000, random_state=SEED)
    model.fit(Xtrain.flatten(1).numpy(), ytrain.numpy())
    pred = model.predict(Xval.flatten(1).numpy())
    score = precision_recall_fscore_support(yval.numpy(),pred,average='macro',zero_division=0)[2]
    baseline_trials.append({'C': C, 'validation_macro_f1': score})
    if score > best_score: baseline, best_score = model, score
display(pd.DataFrame(baseline_trials))
pd.DataFrame(baseline_trials).to_csv(OUT/'baseline_validation.csv',index=False)
joblib.dump(baseline, OUT/'logistic_regression.joblib')

## 5. Improved model: convolutional neural network
Three convolution blocks learn spatial stroke patterns. Dropout and validation-based early stopping limit overfitting. The term “improved model” describes the proposed architecture, not a guaranteed improvement in measured accuracy.

In [ ]:
cnn = nn.Sequential(
    nn.Conv2d(1,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(32,64,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(64,128,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(), nn.Linear(128*4*4,128), nn.ReLU(), nn.Dropout(0.4),
    nn.Linear(128,len(classes))
).to(DEVICE)
loader = DataLoader(torch.utils.data.TensorDataset(Xtrain,ytrain),batch_size=64,shuffle=True,
                    generator=torch.Generator().manual_seed(SEED))
optimizer = torch.optim.Adam(cnn.parameters(), lr=0.001, weight_decay=0.0001)
loss_fn = nn.CrossEntropyLoss()
def predict_cnn(X):
    cnn.eval()
    with torch.no_grad():
        return torch.cat([cnn(batch.to(DEVICE)).argmax(1).cpu() for batch in X.split(128)]).numpy()

history, best_state, best_f1, stale = [], None, -1, 0
for epoch in range(1,31):
    cnn.train(); total_loss = 0
    for xb,yb in loader:
        xb,yb = xb.to(DEVICE),yb.to(DEVICE)
        optimizer.zero_grad(); loss = loss_fn(cnn(xb),yb)
        loss.backward(); optimizer.step(); total_loss += loss.item()*len(xb)
    val_pred = predict_cnn(Xval)
    f1 = precision_recall_fscore_support(yval.numpy(),val_pred,average='macro',zero_division=0)[2]
    history.append({'epoch':epoch,'train_loss':total_loss/len(Xtrain),'val_macro_f1':f1})
    print(history[-1])
    if f1 > best_f1:
        best_f1, best_state, stale = f1, copy.deepcopy(cnn.state_dict()), 0
    else: stale += 1
    if stale >= 6: break
cnn.load_state_dict(best_state)
torch.save({'state_dict':cnn.cpu().state_dict(),'classes':classes}, OUT/'cnn.pt')
cnn.to(DEVICE)
hist = pd.DataFrame(history); hist.to_csv(OUT/'training_history.csv',index=False)
fig,axes = plt.subplots(1,2,figsize=(10,3))
hist.plot(x='epoch',y='train_loss',ax=axes[0]); hist.plot(x='epoch',y='val_macro_f1',ax=axes[1])
plt.tight_layout(); plt.savefig(OUT/'learning_curves.png',dpi=180); plt.show()

## 6. Final held-out evaluation
Run after all model choices are fixed. Report accuracy, macro precision, macro recall, macro F1, per-class scores, and confusion matrices. Do not tune models using these test results.

In [ ]:
Xtest,ytest = tensors(parts['test'])
predictions = {'Logistic regression':baseline.predict(Xtest.flatten(1).numpy()), 'CNN':predict_cnn(Xtest)}
summary = []
for name,pred in predictions.items():
    precision,recall,f1,_ = precision_recall_fscore_support(ytest.numpy(),pred,average='macro',zero_division=0)
    summary.append(dict(model=name,accuracy=accuracy_score(ytest.numpy(),pred),precision_macro=precision,recall_macro=recall,f1_macro=f1))
    report = classification_report(ytest.numpy(),pred,labels=list(range(len(classes))),target_names=classes,zero_division=0,output_dict=True)
    slug = name.lower().replace(' ','_')
    pd.DataFrame(report).T.to_csv(OUT/f'{slug}_classification_report.csv')
    fig,ax = plt.subplots(figsize=(9,8))
    ConfusionMatrixDisplay.from_predictions(ytest.numpy(),pred,labels=list(range(len(classes))),display_labels=classes,ax=ax,cmap='Blues',colorbar=False)
    ax.set_title(name); plt.tight_layout(); plt.savefig(OUT/f'{slug}_confusion_matrix.png',dpi=180); plt.show()
    table = parts['test'].copy(); table['predicted_class'] = [classes[int(i)] for i in pred]
    table.to_csv(OUT/f'{slug}_predictions.csv',index=False)
summary = pd.DataFrame(summary); display(summary)
summary.to_csv(OUT/'model_comparison.csv',index=False)
wrong = np.flatnonzero(predictions['CNN'] != ytest.numpy())[:12]
if len(wrong):
    fig,axes = plt.subplots(3,4,figsize=(10,7))
    for ax in axes.flat: ax.axis('off')
    for ax,i in zip(axes.flat,wrong):
        ax.imshow(Xtest[i,0],cmap='gray')
        ax.set_title(f'True: {classes[ytest[i].item()]} / Pred: {classes[int(predictions["CNN"][i])]}')
    plt.tight_layout(); plt.savefig(OUT/'cnn_errors.png',dpi=180); plt.show()
else: print('No CNN misclassifications in this test split.')

## 7. Discussion to complete after running
- Which model has better test macro F1? Quantify the difference.
- Which classes are most confused? Inspect the saved error images and discuss strokes, crop quality, and handwriting.
- Do training loss and validation F1 suggest overfitting?
- Discuss limited writer diversity, lighting, class ambiguity, and sample size.
- Optional controlled experiment: rerun without augmentation using the SAME split. Compare on validation first; reserve final test evaluation for the chosen experiments.
- Record Python/package versions, hardware, runtime, collection dates, and any changes to the notebook. Save the executed notebook with its outputs.

The report must describe what you actually did. A small test set does not establish general performance or journal readiness.

In [ ]:
import sys, importlib.metadata
versions = {name: importlib.metadata.version(name) for name in ['numpy','pandas','Pillow','matplotlib','scikit-learn','torch','torchvision']}
versions.update(python=sys.version,device=str(DEVICE),seed=SEED)
(OUT/'environment.json').write_text(json.dumps(versions,indent=2),encoding='utf-8')
print('Saved experiment artifacts to', OUT.resolve())